<a href="https://colab.research.google.com/github/BSsangani/How-Data-Quality-Affects-Energy-Demand-Forecasting-in-UK-Smart-Grids/blob/main/step4_modelling_on_degraded_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 4: Running All 4 Models on Every Degraded/Imputed Dataset

This is the step that actually answers RQ1, RQ2, and RQ5. Using the fixed
hyperparameters found in Step 2 (no re-tuning here), I'm running ARIMA,
Linear Regression, Random Forest, and LSTM against:

- the clean baseline (reusing Step 2's numbers directly, since re-running
  it here would just repeat exactly what Step 2 already did)
- the noise and timestamp-shift degraded datasets (no imputation applies
  to these, so they go in as-is)
- all 16 imputed datasets from Step 3 (4 missing-type defects x 4
  imputation methods)

That's 18 runs, each training all 4 models. ARIMA is the slow one (~2-3
minutes per dataset due to one-step-ahead evaluation), so this whole
notebook can take over an hour. Results are saved after every single
dataset, and the notebook skips anything already done -- so if the
session disconnects partway through, just reconnect and re-run this same
notebook; it'll pick up where it left off instead of starting over.

## 0. Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/energy-project')

print("Current folder:", os.getcwd())
print("Files in output:", os.listdir('output'))
print("Degraded files:", os.listdir('output/degraded'))
print("Imputed files (first 5):", os.listdir('output/imputed')[:5])

Mounted at /content/drive
Current folder: /content/drive/MyDrive/energy-project
Files in output: ['clean_demand_2024_2025.csv', 'best_hyperparameters.json', 'degraded', 'imputed', 'imputation_recovery_summary.csv', 'step4_model_comparison.csv']
Degraded files: ['missing_5pct.csv', 'missing_15pct.csv', 'missing_30pct.csv', 'block_gap.csv', 'gaussian_noise.csv', 'timestamp_shift.csv']
Imputed files (first 5): ['missing_5pct__mean_fill.csv', 'missing_5pct__forward_fill.csv', 'missing_5pct__interpolation.csv', 'missing_5pct__kalman_filter.csv', 'missing_15pct__mean_fill.csv']


## 1. Installs

In [ ]:
!pip install -q pandas numpy scikit-learn statsmodels

## 2. Load Step 2's tuned hyperparameters

In [ ]:
import json

with open("output/best_hyperparameters.json") as f:
    hyperparams = json.load(f)

print(json.dumps(hyperparams, indent=2))

{
  "random_forest": {
    "n_estimators": 500,
    "max_depth": 30
  },
  "linear_regression": "no hyperparameters to tune",
  "arima_order": [
    2,
    1,
    2
  ],
  "lstm": {
    "units": 64,
    "dropout": 0.1
  },
  "validation_strategy": "walk-forward (rolling-origin)",
  "baseline_rmse": {
    "random_forest": 402.82,
    "linear_regression": 389.8,
    "arima": 486.52,
    "lstm": 445.8833322056434
  }
}


## 3. The same evaluation functions from Step 2

Exactly the same walk-forward splitter, lag features, and one-step-ahead
ARIMA evaluation as before -- just reused here rather than re-tuning.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

np.random.seed(42)
N_LAGS = 48


def walk_forward_splits(n, n_splits=3, min_train_ratio=0.5):
    min_train = int(n * min_train_ratio)
    test_size = (n - min_train) // n_splits
    for i in range(n_splits):
        train_end = min_train + i * test_size
        test_end = train_end + test_size
        if test_end > n:
            break
        yield np.arange(0, train_end), np.arange(train_end, test_end)


def create_lag_features(series, n_lags):
    X, y = [], []
    for i in range(n_lags, len(series)):
        X.append(series[i - n_lags:i])
        y.append(series[i])
    return np.array(X), np.array(y)


def evaluate_arima(series, order, n_splits=3):
    fold_rmses = []
    for train_idx, test_idx in walk_forward_splits(len(series), n_splits=n_splits):
        train, test = series[train_idx], series[test_idx]
        model = ARIMA(train, order=order).fit()
        preds = []
        for true_val in test:
            preds.append(model.forecast(steps=1)[0])
            model = model.append([true_val], refit=False)
        fold_rmses.append(np.sqrt(mean_squared_error(test, preds)))
    return float(np.mean(fold_rmses))


def evaluate_linear_regression(X, y, n_splits=3):
    fold_rmses = []
    for train_idx, test_idx in walk_forward_splits(len(X), n_splits=n_splits):
        model = LinearRegression()
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[test_idx])
        fold_rmses.append(np.sqrt(mean_squared_error(y[test_idx], preds)))
    return float(np.mean(fold_rmses))


def evaluate_random_forest(X, y, params, n_splits=3):
    fold_rmses = []
    for train_idx, test_idx in walk_forward_splits(len(X), n_splits=n_splits):
        model = RandomForestRegressor(
            n_estimators=params["n_estimators"], max_depth=params["max_depth"],
            random_state=42, n_jobs=-1
        )
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[test_idx])
        fold_rmses.append(np.sqrt(mean_squared_error(y[test_idx], preds)))
    return float(np.mean(fold_rmses))


def evaluate_lstm(series, params, n_lags=N_LAGS, n_splits=2, epochs=5):
    import tensorflow as tf
    tf.random.set_seed(42)
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout

    X, y = create_lag_features(series, n_lags)
    X = X.reshape(-1, n_lags, 1)
    mean, std = y.mean(), y.std()
    X_norm = (X - mean) / std
    y_norm = (y - mean) / std

    fold_rmses = []
    for train_idx, test_idx in walk_forward_splits(len(X_norm), n_splits=n_splits):
        model = Sequential([
            LSTM(params["units"], input_shape=(n_lags, 1)),
            Dropout(params["dropout"]),
            Dense(1),
        ])
        model.compile(optimizer="adam", loss="mse")
        model.fit(X_norm[train_idx], y_norm[train_idx], epochs=epochs, batch_size=64, verbose=0)
        preds_norm = model.predict(X_norm[test_idx], verbose=0).flatten()
        preds = preds_norm * std + mean
        fold_rmses.append(np.sqrt(mean_squared_error(y[test_idx], preds)))
    return float(np.mean(fold_rmses))


def evaluate_all_models(series, hyperparams):
    X_all, y_all = create_lag_features(series, N_LAGS)
    return {
        "arima": evaluate_arima(series, order=tuple(hyperparams["arima_order"])),
        "linear_regression": evaluate_linear_regression(X_all, y_all),
        "random_forest": evaluate_random_forest(X_all, y_all, hyperparams["random_forest"]),
        "lstm": evaluate_lstm(series, hyperparams["lstm"]),
    }

print("Evaluation functions ready.")

Evaluation functions ready.


## 4. The list of scenarios to test

Clean baseline is reused from Step 2 rather than re-run. Noise and
timestamp-shift go in directly. The 16 imputed datasets cover every
defect x imputation combination from Step 3.

In [ ]:
scenarios = [
    {"defect": "clean", "imputation": "none", "path": None},  # reused from Step 2
    {"defect": "gaussian_noise", "imputation": "none", "path": "output/degraded/gaussian_noise.csv"},
    {"defect": "timestamp_shift", "imputation": "none", "path": "output/degraded/timestamp_shift.csv"},
]

for defect in ["missing_5pct", "missing_15pct", "missing_30pct", "block_gap"]:
    for method in ["mean_fill", "forward_fill", "interpolation", "kalman_filter"]:
        scenarios.append({
            "defect": defect,
            "imputation": method,
            "path": f"output/imputed/{defect}__{method}.csv",
        })

print(f"{len(scenarios)} scenarios queued.")

19 scenarios queued.


## 5. Run everything (resumable)

Results are saved after every scenario. If this cell stops partway
(disconnect, timeout, anything), just re-run it -- it skips scenarios
that are already in the results file.

In [8]:
RESULTS_PATH = "output/step4_model_comparison.csv"

# Step 2's own clean-data numbers, reused rather than recomputed
CLEAN_BASELINE = {
    "arima": hyperparams["baseline_rmse"]["arima"],
    "linear_regression": hyperparams["baseline_rmse"]["linear_regression"],
    "random_forest": hyperparams["baseline_rmse"]["random_forest"],
    "lstm": hyperparams["baseline_rmse"]["lstm"],
}

# Load existing results if this is a resume
if os.path.exists(RESULTS_PATH):
    results_df = pd.read_csv(RESULTS_PATH)
    done_keys = set(zip(results_df["defect"], results_df["imputation"]))
    print(f"Resuming: {len(done_keys)} scenario(s) already done.")
else:
    results_df = pd.DataFrame(columns=["defect", "imputation", "arima", "linear_regression", "random_forest", "lstm"])
    done_keys = set()

for i, scenario in enumerate(scenarios):
    key = (scenario["defect"], scenario["imputation"])
    if key in done_keys:
        print(f"[{i+1}/{len(scenarios)}] {key} -- already done, skipping")
        continue

    print(f"[{i+1}/{len(scenarios)}] Running {key} ...")

    if scenario["path"] is None:
        model_rmses = CLEAN_BASELINE
    else:
        df = pd.read_csv(scenario["path"])
        series = df["ND"].values.astype(float)
        model_rmses = evaluate_all_models(series, hyperparams)

    row = {"defect": scenario["defect"], "imputation": scenario["imputation"], **model_rmses}
    results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
    results_df.to_csv(RESULTS_PATH, index=False)

    print(f"    -> {model_rmses}")

print("\nAll scenarios complete.")

Resuming: 19 scenario(s) already done.
[1/19] ('clean', 'none') -- already done, skipping
[2/19] ('gaussian_noise', 'none') -- already done, skipping
[3/19] ('timestamp_shift', 'none') -- already done, skipping
[4/19] ('missing_5pct', 'mean_fill') -- already done, skipping
[5/19] ('missing_5pct', 'forward_fill') -- already done, skipping
[6/19] ('missing_5pct', 'interpolation') -- already done, skipping
[7/19] ('missing_5pct', 'kalman_filter') -- already done, skipping
[8/19] ('missing_15pct', 'mean_fill') -- already done, skipping
[9/19] ('missing_15pct', 'forward_fill') -- already done, skipping
[10/19] ('missing_15pct', 'interpolation') -- already done, skipping
[11/19] ('missing_15pct', 'kalman_filter') -- already done, skipping
[12/19] ('missing_30pct', 'mean_fill') -- already done, skipping
[13/19] ('missing_30pct', 'forward_fill') -- already done, skipping
[14/19] ('missing_30pct', 'interpolation') -- already done, skipping
[15/19] ('missing_30pct', 'kalman_filter') -- already d

## 6. Look at the full results table

In [7]:
results_df = pd.read_csv(RESULTS_PATH)
results_df.round(2)

,defect,imputation,arima,linear_regression,random_forest,lstm
0,clean,none,486.52,389.80,402.82,445.88
1,gaussian_noise,none,993.18,854.25,826.60,857.48
2,timestamp_shift,none,1448.25,1278.67,1193.60,1260.42
3,missing_5pct,mean_fill,1944.51,1741.77,1503.03,1650.74
4,missing_5pct,forward_fill,616.43,512.87,501.31,558.18
5,missing_5pct,interpolation,477.34,385.80,399.94,454.51
6,missing_5pct,kalman_filter,616.61,513.05,501.73,576.25
7,missing_15pct,mean_fill,2814.15,2562.42,2375.35,2483.10
8,missing_15pct,forward_fill,790.70,675.07,633.12,697.32
9,missing_15pct,interpolation,459.23,374.91,392.90,434.67


## 7. Everything's saved to Drive already

`output/step4_model_comparison.csv` now has the RMSE of all 4 models
against the clean baseline, both degraded-only datasets, and all 16
imputed datasets. This is the table the RQ1, RQ2, RQ3, RQ4, and RQ5
answers get built from.